In [16]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Day 15 libraries imported successfully.")

Day 15 libraries imported successfully.


In [17]:
day15_input_dir = Path(
    "../data/analysis_data/"
)

day15_output_dir = Path(
    "../data/analysis_data/"
)

if not day15_input_dir.exists():
    raise FileNotFoundError(
        f"Analysis data directory not found: "
        f"{day15_input_dir.resolve()}"
    )

if not day15_output_dir.exists():
    raise FileNotFoundError(
        f"Output directory not found: "
        f"{day15_output_dir.resolve()}"
    )

print("Day 15 input directory:", day15_input_dir.resolve())
print("Day 15 output directory:", day15_output_dir.resolve())

Day 15 input directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data
Day 15 output directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data


In [18]:
day14_required_files = [
    "APL_Day14_Correlation_Matrix.csv",
    "APL_Day14_Delay_Financial_Relationship.csv",
    "APL_Day14_Discount_Profit_Relationship.csv",
    "APL_Day14_Quantity_Financial_Relationship.csv",
    "APL_Day14_Shipping_Mode_Relationship.csv",
    "APL_Day14_Risk_Delivery_Counts.csv",
    "APL_Day14_Risk_Delivery_Percentages.csv",
    "APL_Day14_Shipping_Duration_Relationship.csv",
    "APL_Day14_Relationship_KPIs.csv"
]

day14_file_validation = pd.DataFrame({
    "Filename": day14_required_files,
    "Exists": [
        (day15_input_dir / filename).is_file()
        for filename in day14_required_files
    ]
})

display(day14_file_validation)

missing_day14_files = day14_file_validation.loc[
    ~day14_file_validation["Exists"],
    "Filename"
].tolist()

if missing_day14_files:
    raise FileNotFoundError(
        f"Missing Day 14 outputs: {missing_day14_files}"
    )

print("All required Day 14 output files are available.")

,Filename,Exists
0,APL_Day14_Correlation_Matrix.csv,True
1,APL_Day14_Delay_Financial_Relationship.csv,True
2,APL_Day14_Discount_Profit_Relationship.csv,True
3,APL_Day14_Quantity_Financial_Relationship.csv,True
4,APL_Day14_Shipping_Mode_Relationship.csv,True
5,APL_Day14_Risk_Delivery_Counts.csv,True
6,APL_Day14_Risk_Delivery_Percentages.csv,True
7,APL_Day14_Shipping_Duration_Relationship.csv,True
8,APL_Day14_Relationship_KPIs.csv,True


All required Day 14 output files are available.


In [19]:
day14_kpis = pd.read_csv(
    day15_input_dir / "APL_Day14_Relationship_KPIs.csv",
    encoding="utf-8-sig"
)

display(day14_kpis)

,Metric,Value
0,Total Shipment Records,180519
1,Overall Actual Delay Rate (%),57.279289160697765
2,Average Shipping Delay Days,0.5658074773292562
3,Average Recorded Order Profit,21.974988616156747
4,Average Profit Margin,0.1083261183756241
5,Shipping Mode with Highest Delay Rate,First Class
6,Highest Shipping Mode Delay Rate (%),100.0
7,Discount Level with Lowest Average Profit Margin,High Discount
8,Lowest Average Profit Margin,0.09522186403948248


## Load the Available Previous-Day KPI Summaries

In [20]:
previous_kpi_files = [
    "APL_Day7_KPIs.csv",
    "APL_Day14_Relationship_KPIs.csv"
]

previous_kpi_data = {}

for filename in previous_kpi_files:

    file_path = day15_input_dir / filename

    if file_path.is_file():

        previous_kpi_data[filename] = pd.read_csv(
            file_path,
            encoding="utf-8-sig"
        )

        print(
            f"Loaded: {filename}",
            previous_kpi_data[filename].shape
        )

    else:

        print(
            f"Not available; will not use: {filename}"
        )

for filename, data in previous_kpi_data.items():

    print(f"\n--- {filename} ---")
    display(data.head())

Loaded: APL_Day7_KPIs.csv (10, 2)
Loaded: APL_Day14_Relationship_KPIs.csv (9, 2)

--- APL_Day7_KPIs.csv ---


,Metric,Value
0,Total Shipment Records,180519
1,Unique Delivery Statuses,4
2,Unique Order Statuses,9
3,Missing Delivery Status Records,0
4,Missing Order Status Records,0



--- APL_Day14_Relationship_KPIs.csv ---


,Metric,Value
0,Total Shipment Records,180519
1,Overall Actual Delay Rate (%),57.279289160697765
2,Average Shipping Delay Days,0.5658074773292562
3,Average Recorded Order Profit,21.974988616156747
4,Average Profit Margin,0.1083261183756241


## Creating the Master KPI Summary

In [21]:
day15_kpi_parts = []

for filename, data in previous_kpi_data.items():

    if {"Metric", "Value"}.issubset(data.columns):

        temp = data[["Metric", "Value"]].copy()

        temp["Source"] = filename

        day15_kpi_parts.append(temp)

if not day15_kpi_parts:

    raise ValueError(
        "No compatible KPI tables were found. "
        "Inspect the available KPI files and their columns."
    )

day15_master_kpis = pd.concat(
    day15_kpi_parts,
    ignore_index=True
)

display(day15_master_kpis)

,Metric,Value,Source
0,Total Shipment Records,180519,APL_Day7_KPIs.csv
1,Unique Delivery Statuses,4,APL_Day7_KPIs.csv
2,Unique Order Statuses,9,APL_Day7_KPIs.csv
3,Missing Delivery Status Records,0,APL_Day7_KPIs.csv
4,Missing Order Status Records,0,APL_Day7_KPIs.csv
5,Overall Actual Delay Rate (%),57.28,APL_Day7_KPIs.csv
6,Most Common Delivery Status,Late delivery,APL_Day7_KPIs.csv
7,Most Common Order Status,COMPLETE,APL_Day7_KPIs.csv
8,Highest Delay Rate Delivery Status,Late delivery,APL_Day7_KPIs.csv
9,Highest Delay Rate Order Status,PENDING,APL_Day7_KPIs.csv


## Prepare the Delivery Performance Dashboard Data

In [22]:
delivery_dashboard_files = [
    "APL_Day7_Delivery_Status_Analysis.csv",
    "APL_Day7_Delivery_Delay_Analysis.csv",
    "APL_Day7_Delivery_Status_by_Mode.csv"
]

delivery_dashboard_data = {}

for filename in delivery_dashboard_files:

    file_path = day15_input_dir / filename

    if file_path.is_file():

        delivery_dashboard_data[filename] = pd.read_csv(
            file_path,
            encoding="utf-8-sig"
        )

        print(
            f"\nLoaded {filename}: "
            f"{delivery_dashboard_data[filename].shape}"
        )

        display(
            delivery_dashboard_data[filename].head()
        )

    else:

        print(f"File not found: {filename}")


Loaded APL_Day7_Delivery_Status_Analysis.csv: (4, 3)


,Delivery_Status,Total_Shipments,Shipment_Share_%
0,Late delivery,98977,54.83
1,Advance shipping,41592,23.04
2,Shipping on time,32196,17.84
3,Shipping canceled,7754,4.30



Loaded APL_Day7_Delivery_Delay_Analysis.csv: (4, 5)


,Delivery Status,Total_Shipments,Actual_Delayed_Shipments,Avg_Delay_Days,Actual_Delay_Rate_%
0,Advance shipping,41592,0,-1.50,0.00
1,Late delivery,98977,98977,1.62,100.00
2,Shipping canceled,7754,4423,0.57,57.04
3,Shipping on time,32196,0,0.00,0.00



Loaded APL_Day7_Delivery_Status_by_Mode.csv: (4, 5)


,Shipping Mode,Advance shipping,Late delivery,Shipping canceled,Shipping on time
0,First Class,0.0,95.32,4.68,0.00
1,Same Day,0.0,45.74,4.56,49.70
2,Second Class,0.0,76.63,4.00,19.36
3,Standard Class,38.6,38.07,4.27,19.06


## Prepare the Delay-Risk Dashboard Data

In [23]:
risk_counts_path = (
    day15_input_dir /
    "APL_Day14_Risk_Delivery_Counts.csv"
)

risk_percentages_path = (
    day15_input_dir /
    "APL_Day14_Risk_Delivery_Percentages.csv"
)

risk_delivery_counts = pd.read_csv(
    risk_counts_path,
    encoding="utf-8-sig"
)

risk_delivery_percentages = pd.read_csv(
    risk_percentages_path,
    encoding="utf-8-sig"
)

print("Risk-delivery counts:")
display(risk_delivery_counts.head())

print("Risk-delivery percentages:")
display(risk_delivery_percentages.head())

Risk-delivery counts:


,Operational_Risk_Profile,Delayed,Early,On Time
0,High Risk,30967,0,0
1,High Risk - Slower Shipping,68010,0,0
2,Lower Risk,4423,43366,33753


Risk-delivery percentages:


,Operational_Risk_Profile,Delayed,Early,On Time
0,High Risk,100.000000,0.000000,0.000000
1,High Risk - Slower Shipping,100.000000,0.000000,0.000000
2,Lower Risk,5.424199,53.182409,41.393392


## Prepare the Shipping and Operational Efficiency Data

In [24]:
shipping_mode_dashboard = pd.read_csv(
    day15_input_dir /
    "APL_Day14_Shipping_Mode_Relationship.csv",
    encoding="utf-8-sig"
)

shipping_duration_dashboard = pd.read_csv(
    day15_input_dir /
    "APL_Day14_Shipping_Duration_Relationship.csv",
    encoding="utf-8-sig"
)

print("Shipping mode dashboard data:")
display(shipping_mode_dashboard.head())

print("Shipping duration dashboard data:")
display(shipping_duration_dashboard.head())

Shipping mode dashboard data:


,Shipping Mode,Total_Shipments,Delayed_Shipments,Average_Shipping_Delay_Days,Average_Actual_Shipping_Days,Average_Scheduled_Shipping_Days,Delay_Rate_%
0,First Class,27814,27814,1.000000,2.000000,1.0,100.000000
1,Same Day,9737,4657,0.478279,0.478279,0.0,47.827873
2,Second Class,35216,28078,1.990828,3.990828,2.0,79.730804
3,Standard Class,107752,42851,-0.004093,3.995907,4.0,39.768171


Shipping duration dashboard data:


,Shipping Mode,Total_Shipments,Average_Actual_Days,Average_Scheduled_Days,Average_Delay_Days,Average_Duration_Gap
0,First Class,27814,2.000000,1.0,1.000000,1.000000
1,Same Day,9737,0.478279,0.0,0.478279,0.478279
2,Second Class,35216,3.990828,2.0,1.990828,1.990828
3,Standard Class,107752,3.995907,4.0,-0.004093,-0.004093


## Prepare the Financial Performance Dashboard Data

In [25]:
financial_dashboard_files = [
    "APL_Day14_Delay_Financial_Relationship.csv",
    "APL_Day14_Discount_Profit_Relationship.csv",
    "APL_Day14_Quantity_Financial_Relationship.csv"
]

financial_dashboard_data = {}

for filename in financial_dashboard_files:

    file_path = day15_input_dir / filename

    financial_dashboard_data[filename] = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

    print(f"\n{filename}")

    display(
        financial_dashboard_data[filename].head()
    )


APL_Day14_Delay_Financial_Relationship.csv


,Delay_Severity,Shipment_Records,Average_Delay_Days,Average_Sales,Total_Sales,Average_Order_Profit,Total_Order_Profit,Average_Profit_Margin
0,Major Delay,42753,2.491615,203.404845,8696167.32,21.082408,901336.20,0.106426
1,Minor Delay,60647,1.000000,203.291464,12329017.44,21.940582,1330630.45,0.107599
2,On Time / Early,77119,-0.843268,204.353655,15759549.55,22.496873,1734936.32,0.109952



APL_Day14_Discount_Profit_Relationship.csv


,Discount_Level,Shipment_Records,Average_Discount_Rate,Average_Sales,Average_Order_Profit,Total_Order_Profit,Average_Profit_Margin
0,High Discount,10029,0.250000,203.841095,19.134183,191896.72,0.095222
1,Low Discount,90259,0.052223,203.742538,23.246366,2098193.71,0.115248
2,Medium Discount,70203,0.158571,203.815294,20.076067,1409400.14,0.098591
3,No Discount,10028,0.000000,203.666647,26.666574,267412.40,0.127283



APL_Day14_Quantity_Financial_Relationship.csv


,Order Item Quantity,Shipment_Records,Average_Product_Price,Average_Sales,Average_Order_Item_Total,Average_Order_Profit,Average_Profit_Margin
0,1,99134,211.664582,211.664582,190.195442,22.667220,0.107478
1,2,20315,55.294771,110.589542,99.394612,12.238763,0.110435
2,3,20350,55.563872,166.691617,149.778986,18.149849,0.108842
3,4,20335,55.423649,221.694597,199.220905,24.162163,0.110062
4,5,20385,55.478215,277.391074,249.262009,29.948165,0.108103


## Prepare the Regional, Market and Country Dashboard Data

In [26]:
available_analysis_files = sorted(
    path.name
    for path in day15_input_dir.glob("*.csv")
)

display(
    pd.DataFrame({
        "Available_Analysis_Files": available_analysis_files
    })
)

,Available_Analysis_Files
0,APL_Day10_Country_Delay_Analysis.csv
1,APL_Day10_Country_Delay_Severity.csv
2,APL_Day10_Country_Delay_Severity_Percentages.csv
3,APL_Day10_Country_Delayed_Duration.csv
4,APL_Day10_Country_KPIs.csv
5,APL_Day10_Country_Performance.csv
6,APL_Day10_Country_Risk_Table.csv
7,APL_Day10_Country_Shipment_Summary.csv
8,APL_Day11_Customer_Segment_Delay_Analysis.csv
9,APL_Day11_Customer_Segment_Distribution.csv


## Prepare the Customer and Product Dashboard Data

In [29]:
# ============================================================
# CUSTOMER SEGMENT & PRODUCT/CATEGORY DASHBOARD DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
#  DEFINE INPUT AND OUTPUT PATHS
# ------------------------------------------------------------

day15_input_dir = Path("../data/analysis_data/")
day15_feature_dir = Path("../data/feature_data/")
day15_output_dir = Path("../data/analysis_data/")

day15_output_dir.mkdir(parents=True, exist_ok=True)

# Detailed source dataset used in the Day 11 and Day 13 analyses
day15_source_file = (
    day15_input_dir / "APL_Day5_Delay_Risk_Analysis.csv"
)

# Fallback source if the Day 5 analysis dataset is unavailable
day15_fallback_file = (
    day15_feature_dir / "APL_Day3_Feature_Engineered.csv"
)

print("Day 15 Input Directory:", day15_input_dir.resolve())
print("Day 15 Output Directory:", day15_output_dir.resolve())


# ------------------------------------------------------------
#  CHECK DAY 11 CUSTOMER SEGMENT OUTPUTS
# ------------------------------------------------------------

day11_files = {
    "Customer Segment Distribution":
        "APL_Day11_Customer_Segment_Distribution.csv",

    "Customer Segment Shipment Analysis":
        "APL_Day11_Customer_Segment_Shipment_Analysis.csv",

    "Customer Segment Financial Analysis":
        "APL_Day11_Customer_Segment_Financial_Analysis.csv",

    "Customer Segment Delay Analysis":
        "APL_Day11_Customer_Segment_Delay_Analysis.csv",

    "Customer Segment Risk Analysis":
        "APL_Day11_Customer_Segment_Risk_Analysis.csv",

    "Customer Segment KPIs":
        "APL_Day11_Customer_Segment_KPIs.csv"
}

day11_file_status = []

for dataset_name, filename in day11_files.items():

    file_path = day15_input_dir / filename

    day11_file_status.append({
        "Dataset": dataset_name,
        "Filename": filename,
        "File_Exists": file_path.exists(),
        "Path": str(file_path)
    })

day11_file_status = pd.DataFrame(day11_file_status)

print("\nDay 11 Dataset Availability:")
display(day11_file_status)


# ------------------------------------------------------------
#  CHECK DAY 13 PRODUCT/CATEGORY OUTPUTS
# ------------------------------------------------------------

day13_files = {
    "Category Performance":
        "APL_Day13_Category_Performance.csv",

    "Product Performance":
        "APL_Day13_Product_Performance.csv",

    "Department Performance":
        "APL_Day13_Department_Performance.csv",

    "Category Delay Severity":
        "APL_Day13_Category_Delay_Severity.csv",

    "Product Delay Analysis":
        "APL_Day13_Product_Delay_Analysis.csv",

    "Negative Profit Category":
        "APL_Day13_Negative_Profit_Category.csv",

    "Negative Profit Product":
        "APL_Day13_Negative_Profit_Product.csv",

    "Product Category KPIs":
        "APL_Day13_Product_Category_KPIs.csv",

    "Day 13 Validation":
        "APL_Day13_Validation.csv"
}

day13_file_status = []

for dataset_name, filename in day13_files.items():

    file_path = day15_input_dir / filename

    day13_file_status.append({
        "Dataset": dataset_name,
        "Filename": filename,
        "File_Exists": file_path.exists(),
        "Path": str(file_path)
    })

day13_file_status = pd.DataFrame(day13_file_status)

print("\nDay 13 Dataset Availability:")
display(day13_file_status)


# ------------------------------------------------------------
#  LOAD THE DETAILED SOURCE DATASET
# ------------------------------------------------------------

if day15_source_file.exists():

    source_path = day15_source_file

elif day15_fallback_file.exists():

    source_path = day15_fallback_file

else:

    raise FileNotFoundError(
        "Neither source dataset was found.\n"
        f"Checked: {day15_source_file.resolve()}\n"
        f"Checked: {day15_fallback_file.resolve()}"
    )

df_day15_customer_product = pd.read_csv(
    source_path,
    encoding="cp1252",
    low_memory=False
)

print("\nSelected Source Dataset:", source_path.resolve())
print("Source Dataset Shape:", df_day15_customer_product.shape)

print("\nSource Dataset Columns:")
display(
    pd.DataFrame({
        "Column": df_day15_customer_product.columns
    })
)


# ------------------------------------------------------------
#  STANDARDIZE COLUMN NAMES USED IN THE ANALYSIS
# ------------------------------------------------------------

# Support common naming variations without changing the
# original column names in the source dataframe.

column_aliases = {
    "Customer Segment": [
        "Customer Segment",
        "Customer_Segment",
        "Customer Segment Name"
    ],

    "Category Name": [
        "Category Name",
        "Category_Name",
        "Product Category",
        "Product_Category"
    ],

    "Product Name": [
        "Product Name",
        "Product_Name"
    ],

    "Department Name": [
        "Department Name",
        "Department_Name"
    ],

    "Sales": [
        "Sales"
    ],

    "Order Item Total": [
        "Order Item Total",
        "Order_Item_Total"
    ],

    "Order Profit Per Order": [
        "Order Profit Per Order",
        "Order_Profit_Per_Order"
    ],

    "Order Item Quantity": [
        "Order Item Quantity",
        "Order_Item_Quantity"
    ],

    "Product Price": [
        "Product Price",
        "Product_Price"
    ],

    "Late_delivery_risk": [
        "Late_delivery_risk",
        "Late Delivery Risk"
    ],

    "Is_Delayed": [
        "Is_Delayed",
        "Is Delayed"
    ],

    "Days for shipping (real)": [
        "Days for shipping (real)",
        "Days_for_shipping_real"
    ],

    "Shipping_Delay_Days": [
        "Shipping_Delay_Days",
        "Shipping Delay Days"
    ]
}

rename_mapping = {}

for standard_name, possible_names in column_aliases.items():

    for possible_name in possible_names:

        if possible_name in df_day15_customer_product.columns:

            if possible_name != standard_name:
                rename_mapping[possible_name] = standard_name

            break

df_day15_customer_product = (
    df_day15_customer_product.rename(columns=rename_mapping)
)

print("\nStandardized Column Names:")
display(
    pd.DataFrame({
        "Available Columns":
            df_day15_customer_product.columns
    })
)


# ------------------------------------------------------------
#  VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

customer_required_columns = [
    "Customer Segment",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Item Quantity",
    "Late_delivery_risk",
    "Is_Delayed"
]

product_required_columns = [
    "Category Name",
    "Product Name",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Item Quantity",
    "Late_delivery_risk",
    "Is_Delayed"
]

available_columns = set(
    df_day15_customer_product.columns
)

missing_customer_columns = [
    column for column in customer_required_columns
    if column not in available_columns
]

missing_product_columns = [
    column for column in product_required_columns
    if column not in available_columns
]

print("\nMissing Customer Dashboard Columns:")
print(missing_customer_columns)

print("\nMissing Product Dashboard Columns:")
print(missing_product_columns)

if missing_customer_columns:

    raise ValueError(
        "Required customer dashboard columns are missing: "
        f"{missing_customer_columns}"
    )

if missing_product_columns:

    raise ValueError(
        "Required product dashboard columns are missing: "
        f"{missing_product_columns}"
    )


# ------------------------------------------------------------
#  PREPARE NUMERIC FIELDS
# ------------------------------------------------------------

numeric_columns = [
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Item Quantity",
    "Late_delivery_risk",
    "Is_Delayed",
    "Product Price",
    "Days for shipping (real)",
    "Shipping_Delay_Days"
]

for column in numeric_columns:

    if column in df_day15_customer_product.columns:

        df_day15_customer_product[column] = pd.to_numeric(
            df_day15_customer_product[column],
            errors="coerce"
        )


# Preserve the number of source records before grouping.
source_record_count = len(df_day15_customer_product)

# Missing segment/category/product names are explicitly labelled.
# This prevents records from disappearing during groupby.
for column in [
    "Customer Segment",
    "Category Name",
    "Product Name"
]:

    if column in df_day15_customer_product.columns:

        df_day15_customer_product[column] = (
            df_day15_customer_product[column]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
            .replace("", "Unknown")
        )


# ------------------------------------------------------------
#  HELPER FUNCTION FOR DASHBOARD METRICS
# ------------------------------------------------------------

def build_dashboard_summary(
    dataframe,
    group_columns,
    include_product_price=False
):
    """
    Create dashboard-ready aggregated data.

    Counts:
        Number of source records

    Financial measures:
        Sum of Sales
        Sum of Order Item Total
        Sum of Order Profit Per Order

    Operational measures:
        Total Order Item Quantity
        High-risk record count and rate
        Actual delayed record count and rate

    Optional:
        Average Product Price
        Average actual shipping duration
        Average recorded shipping delay
    """

    work_df = dataframe.copy()

    # --------------------------------------------------------
    # Validate grouping fields
    # --------------------------------------------------------

    missing_group_columns = [
        column for column in group_columns
        if column not in work_df.columns
    ]

    if missing_group_columns:

        raise ValueError(
            f"Missing grouping columns: {missing_group_columns}"
        )

    # --------------------------------------------------------
    # Group by the requested dashboard dimensions
    # --------------------------------------------------------

    grouped = work_df.groupby(
        group_columns,
        dropna=False,
        observed=True
    )

    summary = grouped.size().rename(
        "Shipment_Record_Count"
    ).to_frame()

    # --------------------------------------------------------
    # Financial and quantity measures
    # --------------------------------------------------------

    financial_columns = [
        "Sales",
        "Order Item Total",
        "Order Profit Per Order",
        "Order Item Quantity"
    ]

    for column in financial_columns:

        if column in work_df.columns:

            summary[f"Total_{column.replace(' ', '_')}"] = (
                grouped[column].sum(min_count=1)
            )

    # --------------------------------------------------------
    # Late delivery risk
    # --------------------------------------------------------

    if "Late_delivery_risk" in work_df.columns:

        summary["High_Risk_Record_Count"] = (
            grouped["Late_delivery_risk"].sum(min_count=1)
        )

        summary["High_Risk_Rate_%"] = (
            grouped["Late_delivery_risk"].mean() * 100
        )

    # --------------------------------------------------------
    # Actual delay
    # --------------------------------------------------------

    if "Is_Delayed" in work_df.columns:

        summary["Delayed_Record_Count"] = (
            grouped["Is_Delayed"].sum(min_count=1)
        )

        summary["Actual_Delay_Rate_%"] = (
            grouped["Is_Delayed"].mean() * 100
        )

    # --------------------------------------------------------
    # Optional product pricing measure
    # --------------------------------------------------------

    if (
        include_product_price
        and "Product Price" in work_df.columns
    ):

        summary["Average_Product_Price"] = (
            grouped["Product Price"].mean()
        )

    # --------------------------------------------------------
    # Optional shipping measures
    # --------------------------------------------------------

    if "Days for shipping (real)" in work_df.columns:

        summary["Average_Actual_Shipping_Days"] = (
            grouped["Days for shipping (real)"].mean()
        )

    if "Shipping_Delay_Days" in work_df.columns:

        summary["Average_Shipping_Delay_Days"] = (
            grouped["Shipping_Delay_Days"].mean()
        )

    # --------------------------------------------------------
    # Convert grouping index back into regular columns
    # --------------------------------------------------------

    summary = summary.reset_index()

    # Round numeric dashboard metrics for readability.
    numeric_summary_columns = summary.select_dtypes(
        include="number"
    ).columns

    summary[numeric_summary_columns] = (
        summary[numeric_summary_columns].round(2)
    )

    return summary


# ------------------------------------------------------------
#  BUILD CUSTOMER SEGMENT DASHBOARD DATA
# ------------------------------------------------------------

customer_segment_dashboard = build_dashboard_summary(
    dataframe=df_day15_customer_product,
    group_columns=["Customer Segment"]
)

customer_segment_dashboard = (
    customer_segment_dashboard.sort_values(
        by="Shipment_Record_Count",
        ascending=False
    ).reset_index(drop=True)
)

customer_dashboard_path = (
    day15_output_dir
    / "APL_Day15_Customer_Segment_Dashboard_Data.csv"
)

customer_segment_dashboard.to_csv(
    customer_dashboard_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCustomer Segment Dashboard Data:")
display(customer_segment_dashboard.head(10))

print(
    "\nCustomer dashboard saved to:",
    customer_dashboard_path.resolve()
)


# ------------------------------------------------------------
#  BUILD PRODUCT/CATEGORY DASHBOARD DATA
# ------------------------------------------------------------

product_category_dashboard = build_dashboard_summary(
    dataframe=df_day15_customer_product,
    group_columns=[
        "Category Name",
        "Product Name"
    ],
    include_product_price=True
)

product_category_dashboard = (
    product_category_dashboard.sort_values(
        by="Shipment_Record_Count",
        ascending=False
    ).reset_index(drop=True)
)

product_dashboard_path = (
    day15_output_dir
    / "APL_Day15_Product_Category_Dashboard_Data.csv"
)

product_category_dashboard.to_csv(
    product_dashboard_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nProduct/Category Dashboard Data:")
display(product_category_dashboard.head(10))

print(
    "\nProduct/category dashboard saved to:",
    product_dashboard_path.resolve()
)


# ------------------------------------------------------------
#  CREATE CUSTOMER/PRODUCT VALIDATION REPORT
# ------------------------------------------------------------

customer_segment_count = (
    df_day15_customer_product["Customer Segment"].nunique()
)

category_count = (
    df_day15_customer_product["Category Name"].nunique()
)

product_count = (
    df_day15_customer_product["Product Name"].nunique()
)

customer_dashboard_record_count = (
    int(customer_segment_dashboard["Shipment_Record_Count"].sum())
)

product_dashboard_record_count = (
    int(product_category_dashboard["Shipment_Record_Count"].sum())
)

validation_rows = [
    {
        "Validation": "Source dataset loaded",
        "Value": str(source_path.resolve()),
        "Status": "PASS"
    },
    {
        "Validation": "Source record count",
        "Value": source_record_count,
        "Status": (
            "PASS" if source_record_count > 0 else "FAIL"
        )
    },
    {
        "Validation": "Unique customer segments",
        "Value": customer_segment_count,
        "Status": (
            "PASS" if customer_segment_count > 0 else "FAIL"
        )
    },
    {
        "Validation": "Unique product categories",
        "Value": category_count,
        "Status": (
            "PASS" if category_count > 0 else "FAIL"
        )
    },
    {
        "Validation": "Unique product names",
        "Value": product_count,
        "Status": (
            "PASS" if product_count > 0 else "FAIL"
        )
    },
    {
        "Validation": "Customer dashboard record reconciliation",
        "Value": (
            f"{customer_dashboard_record_count} / "
            f"{source_record_count}"
        ),
        "Status": (
            "PASS"
            if customer_dashboard_record_count == source_record_count
            else "CHECK"
        )
    },
    {
        "Validation": "Product dashboard record reconciliation",
        "Value": (
            f"{product_dashboard_record_count} / "
            f"{source_record_count}"
        ),
        "Status": (
            "PASS"
            if product_dashboard_record_count == source_record_count
            else "CHECK"
        )
    },
    {
        "Validation": "Customer dashboard file exists",
        "Value": str(customer_dashboard_path.resolve()),
        "Status": (
            "PASS" if customer_dashboard_path.exists() else "FAIL"
        )
    },
    {
        "Validation": "Product dashboard file exists",
        "Value": str(product_dashboard_path.resolve()),
        "Status": (
            "PASS" if product_dashboard_path.exists() else "FAIL"
        )
    },
    {
        "Validation": "Customer dashboard contains rows",
        "Value": len(customer_segment_dashboard),
        "Status": (
            "PASS"
            if not customer_segment_dashboard.empty
            else "FAIL"
        )
    },
    {
        "Validation": "Product dashboard contains rows",
        "Value": len(product_category_dashboard),
        "Status": (
            "PASS"
            if not product_category_dashboard.empty
            else "FAIL"
        )
    }
]

customer_product_validation = pd.DataFrame(validation_rows)

customer_product_validation_path = (
    day15_output_dir
    / "APL_Day15_Customer_Product_Validation.csv"
)

customer_product_validation.to_csv(
    customer_product_validation_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCustomer/Product Dashboard Validation:")
display(customer_product_validation)

print(
    "\nValidation report saved to:",
    customer_product_validation_path.resolve()
)


# ------------------------------------------------------------
#  FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DAY 15 SECTION 11: CUSTOMER & PRODUCT DASHBOARD DATA")
print("=" * 65)

print("Source Dataset:", source_path.name)
print("Source Records:", source_record_count)

print(
    "Customer Segment Dashboard:",
    customer_segment_dashboard.shape
)

print(
    "Product/Category Dashboard:",
    product_category_dashboard.shape
)

print(
    "Customer Segment Output:",
    customer_dashboard_path.name
)

print(
    "Product/Category Output:",
    product_dashboard_path.name
)

print(
    "Validation Output:",
    customer_product_validation_path.name
)

if (
    (customer_product_validation["Status"] == "FAIL").any()
):
    print("\nWARNING: One or more validation checks failed.")

else:
    print("\nSection 11 processing completed.")
    print("Review any CHECK statuses before proceeding.")

print("=" * 65)

Day 15 Input Directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data
Day 15 Output Directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data

Day 11 Dataset Availability:


,Dataset,Filename,File_Exists,Path
0,Customer Segment Distribution,APL_Day11_Customer_Segment_Distribution.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...
1,Customer Segment Shipment Analysis,APL_Day11_Customer_Segment_Shipment_Analysis.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...
2,Customer Segment Financial Analysis,APL_Day11_Customer_Segment_Financial_Analysis.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...
3,Customer Segment Delay Analysis,APL_Day11_Customer_Segment_Delay_Analysis.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...
4,Customer Segment Risk Analysis,APL_Day11_Customer_Segment_Risk_Analysis.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...
5,Customer Segment KPIs,APL_Day11_Customer_Segment_KPIs.csv,True,..\data\analysis_data\APL_Day11_Customer_Segme...



Day 13 Dataset Availability:


,Dataset,Filename,File_Exists,Path
0,Category Performance,APL_Day13_Category_Performance.csv,True,..\data\analysis_data\APL_Day13_Category_Perfo...
1,Product Performance,APL_Day13_Product_Performance.csv,True,..\data\analysis_data\APL_Day13_Product_Perfor...
2,Department Performance,APL_Day13_Department_Performance.csv,True,..\data\analysis_data\APL_Day13_Department_Per...
3,Category Delay Severity,APL_Day13_Category_Delay_Severity.csv,True,..\data\analysis_data\APL_Day13_Category_Delay...
4,Product Delay Analysis,APL_Day13_Product_Delay_Analysis.csv,True,..\data\analysis_data\APL_Day13_Product_Delay_...
5,Negative Profit Category,APL_Day13_Negative_Profit_Category.csv,True,..\data\analysis_data\APL_Day13_Negative_Profi...
6,Negative Profit Product,APL_Day13_Negative_Profit_Product.csv,True,..\data\analysis_data\APL_Day13_Negative_Profi...
7,Product Category KPIs,APL_Day13_Product_Category_KPIs.csv,True,..\data\analysis_data\APL_Day13_Product_Catego...
8,Day 13 Validation,APL_Day13_Validation.csv,False,..\data\analysis_data\APL_Day13_Validation.csv



Selected Source Dataset: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day5_Delay_Risk_Analysis.csv
Source Dataset Shape: (180519, 56)

Source Dataset Columns:


...


Customer dashboard saved to: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day15_Customer_Segment_Dashboard_Data.csv

Product/Category Dashboard Data:


,Category Name,Product Name,Shipment_Record_Count,Total_Sales,Total_Order_Item_Total,Total_Order_Profit_Per_Order,Total_Order_Item_Quantity,High_Risk_Record_Count,High_Risk_Rate_%,Delayed_Record_Count,Actual_Delay_Rate_%,Average_Product_Price,Average_Actual_Shipping_Days,Average_Shipping_Delay_Days
0,Cleats,Perfect Fitness Perfect Rip Deck,24515,4421143.02,3973180.31,493828.30,73698,13473,54.96,14053,57.32,59.99,3.51,0.58
1,Men's Footwear,Nike Men's CJ Elite 2 TD Football Cleat,22246,2891757.54,2598494.44,311902.82,22246,12121,54.49,12664,56.93,129.99,3.50,0.56
2,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,21035,3147800.00,2828708.50,350421.03,62956,11476,54.56,12017,57.13,50.00,3.48,0.56
3,Indoor/Outdoor Games,O'Brien Men's Neoprene Life Vest,19298,2888993.94,2596454.04,318451.43,57803,10565,54.75,11043,57.22,49.98,3.50,0.56
4,Fishing,Field & Stream Sportsman 16 Gun Fire Safe,17325,6929653.50,6226935.12,756220.76,17325,9516,54.93,9930,57.32,399.98,3.50,0.56
5,Water Sports,Pelican Sunstream 100 Kayak,15500,3099845.00,2785518.00,324076.37,15500,8492,54.79,8866,57.20,199.99,3.48,0.57
6,Camping & Hiking,Diamondback Women's Serene Classic Comfort Bi,13729,4118425.42,3700783.54,427455.57,13729,7487,54.53,7812,56.90,299.98,3.50,0.55
7,Cardio Equipment,Nike Men's Free 5.0+ Running Shoe,12169,3667633.20,3295693.32,379915.82,36680,6629,54.47,6925,56.91,99.99,3.49,0.55
8,Shop By Sport,Under Armour Girls' Toddler Spine Surge Runni,10617,1269082.65,1140770.95,126278.51,31735,5865,55.24,6126,57.70,39.99,3.49,0.58
9,Video Games,Fighting video games,838,33310.50,29919.20,2717.52,838,455,54.30,472,56.32,39.75,3.59,0.58



Product/category dashboard saved to: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day15_Product_Category_Dashboard_Data.csv

Customer/Product Dashboard Validation:


,Validation,Value,Status
0,Source dataset loaded,D:\Data Analytics Project\Delivery Performance...,PASS
1,Source record count,180519,PASS
2,Unique customer segments,3,PASS
3,Unique product categories,50,PASS
4,Unique product names,118,PASS
5,Customer dashboard record reconciliation,180519 / 180519,PASS
6,Product dashboard record reconciliation,180519 / 180519,PASS
7,Customer dashboard file exists,D:\Data Analytics Project\Delivery Performance...,PASS
8,Product dashboard file exists,D:\Data Analytics Project\Delivery Performance...,PASS
9,Customer dashboard contains rows,3,PASS



Validation report saved to: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day15_Customer_Product_Validation.csv

DAY 15 SECTION 11: CUSTOMER & PRODUCT DASHBOARD DATA
Source Dataset: APL_Day5_Delay_Risk_Analysis.csv
Source Records: 180519
Customer Segment Dashboard: (3, 12)
Product/Category Dashboard: (118, 14)
Customer Segment Output: APL_Day15_Customer_Segment_Dashboard_Data.csv
Product/Category Output: APL_Day15_Product_Category_Dashboard_Data.csv
Validation Output: APL_Day15_Customer_Product_Validation.csv

Section 11 processing completed.
Review any CHECK statuses before proceeding.


## Build a Dashboard Dataset Inventory

In [34]:
dashboard_inventory = pd.DataFrame({
    "Dashboard_Module": [
        "Executive Overview",
        "Delivery Performance",
        "Delay Risk",
        "Shipping Efficiency",
        "Financial Performance",
        "Regional Analysis",
        "Market Analysis",
        "Country Analysis",
        "Customer Segments",
        "Product and Category",
        "Relationship Analysis"
    ],

    "Purpose": [
        "Summarize the main logistics KPIs",
        "Compare delivery outcomes and delay rates",
        "Examine risk categories and delivery outcomes",
        "Compare shipping modes and delivery duration",
        "Summarize profit, margin and discount relationships",
        "Compare performance across regions",
        "Compare shipment performance across markets",
        "Compare shipment performance across countries",
        "Explore customer-segment performance",
        "Explore product and category performance",
        "Examine relationships between operational and financial variables"
    ],

    "Status": [
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready"
    ]
})

display(dashboard_inventory)

,Dashboard_Module,Purpose,Status
0,Executive Overview,Summarize the main logistics KPIs,Ready
1,Delivery Performance,Compare delivery outcomes and delay rates,Ready
2,Delay Risk,Examine risk categories and delivery outcomes,Ready
3,Shipping Efficiency,Compare shipping modes and delivery duration,Ready
4,Financial Performance,"Summarize profit, margin and discount relation...",Ready
5,Regional Analysis,Compare performance across regions,Ready
6,Market Analysis,Compare shipment performance across markets,Ready
7,Country Analysis,Compare shipment performance across countries,Ready
8,Customer Segments,Explore customer-segment performance,Ready
9,Product and Category,Explore product and category performance,Ready


## Saving Day 15 KPI & Dashboard Outputs

In [35]:
day15_outputs = {
    "APL_Day15_Master_KPIs.csv":
        day15_master_kpis,

    "APL_Day15_Dashboard_Data_Inventory.csv":
        dashboard_inventory
}

for filename, output_df in day15_outputs.items():

    output_path = day15_output_dir / filename

    output_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", output_path.resolve())

print("Day 15 KPI and dashboard inventory outputs saved.")

Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day15_Master_KPIs.csv
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day15_Dashboard_Data_Inventory.csv
Day 15 KPI and dashboard inventory outputs saved.


## Validation for Day 15 Outputs

In [36]:
day15_validation_checks = {
    "Master KPI Table Not Empty":
        not day15_master_kpis.empty,

    "Master KPI Required Columns Available":
        {"Metric", "Value", "Source"}.issubset(
            day15_master_kpis.columns
        ),

    "Dashboard Inventory Not Empty":
        not dashboard_inventory.empty,

    "Dashboard Inventory Required Columns Available":
        {
            "Dashboard_Module",
            "Purpose",
            "Status"
        }.issubset(dashboard_inventory.columns),

    "All Day 15 Output Files Exist":
        all(
            (day15_output_dir / filename).is_file()
            for filename in day15_outputs
        )
}

day15_validation_report = pd.DataFrame({
    "Validation_Check": list(
        day15_validation_checks.keys()
    ),

    "Status": [
        "PASS" if result else "FAIL"
        for result in day15_validation_checks.values()
    ]
})

display(day15_validation_report)

if all(day15_validation_checks.values()):

    print("ALL DAY 15 BASIC VALIDATION CHECKS PASSED.")

else:

    print("SOME DAY 15 VALIDATION CHECKS FAILED.")

,Validation_Check,Status
0,Master KPI Table Not Empty,PASS
1,Master KPI Required Columns Available,PASS
2,Dashboard Inventory Not Empty,PASS
3,Dashboard Inventory Required Columns Available,PASS
4,All Day 15 Output Files Exist,PASS


ALL DAY 15 BASIC VALIDATION CHECKS PASSED.


In [1]:
import pandas as pd

file_path = r"D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\raw_data\APL_Logistics.csv"

df_check = pd.read_csv(
    file_path,
    encoding="cp1252",
    low_memory=False,
    nrows=5
)

print("ALL COLUMN NAMES:")
for col in df_check.columns:
    print(repr(col))

print("\nColumns containing 'date':")
print([
    col for col in df_check.columns
    if "date" in str(col).lower()
])

ALL COLUMN NAMES:
'Type'
'Days for shipping (real)'
'Days for shipment (scheduled)'
'Benefit per order'
'Sales per customer'
'Delivery Status'
'Late_delivery_risk'
'Category Id'
'Category Name'
'Customer City'
'Customer Country'
'Customer Fname'
'Customer Id'
'Customer Lname'
'Customer Segment'
'Customer State'
'Customer Street'
'Customer Zipcode'
'Department Id'
'Department Name'
'Latitude'
'Longitude'
'Market'
'Order City'
'Order Country'
'Order Customer Id'
'Order Item Discount'
'Order Item Discount Rate'
'Order Item Product Price'
'Order Item Profit Ratio'
'Order Item Quantity'
'Sales'
'Order Item Total'
'Order Profit Per Order'
'Order Region'
'Order State'
'Order Status'
'Product Name'
'Product Price'
'Shipping Mode'

Columns containing 'date':
[]
